# SOV33 GRPO Training on Kaggle T4

Train sovereign AI model using GRPO (Group Relative Policy Optimization)
Based on DeepSeek-R1's approach + HuggingFace TRL + Unsloth

**Goal:** Fix sovereign knowledge gap (0% → 80%+) and improve reasoning (75% → 90%+)

**Model:** Qwen/Qwen2.5-3B-Instruct
**GPU:** T4 (16GB VRAM)
**Training:** GRPO with sovereign reward functions

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl peft accelerate transformers datasets

In [ ]:
# Cell 2: Imports
import torch
import json
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Cell 3: Sovereign Knowledge Training Data (25 examples)
SOVEREIGN_DATA = [
    {"prompt": "What is the DEFONEOS care floor value?", "completion": "The DEFONEOS care floor value is 0.95. This is the minimum quality threshold that all outputs must meet. Any output scoring below 0.95 is rejected by the sovereign governance system."},
    {"prompt": "How many agents are in the BFT council?", "completion": "The BFT council consists of 33 agents. A quorum of 23 out of 33 (23/33) is required for binding decisions. The consensus mechanism uses HotStuff."},
    {"prompt": "What cryptographic algorithm does SIGIL use?", "completion": "SIGIL uses Ed25519 cryptographic signatures. Each signature is hash-linked to the previous one, creating an immutable audit chain at 1Hz rate."},
    {"prompt": "What is Article 0 in the sovereign framework?", "completion": "Article 0 is fee-for-service only. It prohibits equity stakes, board seats, and VC money. This ensures the sovereign AI operates purely on service fees."},
    {"prompt": "When does EU AI Act Article 50 take effect?", "completion": "EU AI Act Article 50 takes effect on 2 August 2026. It covers transparency obligations for AI systems."},
    {"prompt": "What is the GDPR breach notification deadline?", "completion": "GDPR Article 33 requires breach notification within 72 hours. Organizations must report personal data breaches to the supervisory authority."},
    {"prompt": "What is the GDPR maximum fine?", "completion": "GDPR Article 83 sets the maximum fine at 20 million euros or 4% of worldwide annual turnover, whichever is higher."},
    {"prompt": "What is ISO 42001?", "completion": "ISO 42001 is the AI Management System (AIMS) standard. It has 7 clauses (4-10) and Annex A, providing a framework for responsible AI governance."},
    {"prompt": "What is AUKUS Pillar 2?", "completion": "AUKUS Pillar 2 is a 2.4 billion dollar program over 5 years. It focuses on AI, autonomy, quantum computing, and cyber capabilities."},
    {"prompt": "What is the NCSC CAF?", "completion": "The NCSC CAF v3.1 has 14 security outcomes across 4 objectives. It provides a framework for assessing cybersecurity in AI systems."},
    {"prompt": "What is the CSOAI Companies House number?", "completion": "CSOAI Ltd is registered at UK Companies House with number 16939677."},
    {"prompt": "What are the 5 OWEM groups?", "completion": "The 5 OWEM groups are: compliance, defense, intuition, voice, and general. Each group specializes in different aspects of sovereign AI governance."},
    {"prompt": "List the 12 Pillars of the sovereign framework.", "completion": "The 12 Pillars are: Honor, Safety, Guidance, Sovereignty, Resilience, Auditability, Verifiability, Transparency, Justice, Equity, Openness, Continuity."},
    {"prompt": "What are the 7 Red Lines?", "completion": "The 7 Red Lines are: No kinetic targeting, no surveillance, no civilian harm, no sovereignty violations, no auto-escalation, no lying, no irreversibility."},
    {"prompt": "What is the sovereign DID?", "completion": "The sovereign DID is did:csoai:nicholas-001. It is the decentralized identifier for the sovereign AI operator."},
    # Reasoning
    {"prompt": "A bat and a ball cost $1.10. The bat costs $1.00 more than the ball. How much does the ball cost?", "completion": "Let B = ball cost. Then B + (B + 1.00) = 1.10. So 2B = 0.10, B = 0.05. The ball costs $0.05."},
    {"prompt": "If all roses are flowers, and some flowers fade quickly, can we conclude that some roses fade quickly?", "completion": "No. This is a logical fallacy. The fact that some flowers fade does not tell us which specific flowers fade. We cannot conclude that roses are among them."},
    {"prompt": "Can you catch a cold from being cold?", "completion": "No. Colds are caused by viruses (rhinovirus), not by cold temperatures. Being cold does not cause a cold."},
    {"prompt": "What is 15% of 200?", "completion": "15% of 200 = 0.15 x 200 = 30."},
    {"prompt": "What is 2^10?", "completion": "2^10 = 1024."},
    {"prompt": "What is 7! (7 factorial)?", "completion": "7! = 7 x 6 x 5 x 4 x 3 x 2 x 1 = 5040."},
    {"prompt": "A farmer has 17 sheep. All but 9 die. How many are left?", "completion": "9 sheep are left. All but 9 die means 9 survive."},
    {"prompt": "If 3 workers can build a wall in 6 days, how many days for 9 workers?", "completion": "This is inverse proportion. 3 workers x 6 days = 18 worker-days. 9 workers x ? days = 18 worker-days. So ? = 18/9 = 2 days."},
    # Code
    {"prompt": "Write a Python function is_palindrome(s).", "completion": "def is_palindrome(s):\n    s = s.lower().strip()\n    return s == s[::-1]"},
    {"prompt": "Write a Python function factorial(n).", "completion": "def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)"},
]

print(f"Training examples: {len(SOVEREIGN_DATA)}")

In [ ]:
# Cell 4: Format data for SFT
formatted_data = []
for item in SOVEREIGN_DATA:
    formatted_data.append({
        "messages": [
            {"role": "system", "content": "You are SOV33, a sovereign AI model. Answer concisely and accurately."},
            {"role": "user", "content": item["prompt"]},
            {"role": "assistant", "content": item["completion"]},
        ]
    })

dataset = Dataset.from_list(formatted_data)
print(f"Dataset size: {len(dataset)}")
print(f"Sample: {dataset[0]['messages'][1]['content'][:50]}...")

In [ ]:
# Cell 5: Load model with Unsloth for 2x speedup
from unsloth import FastLanguageModel

model_name = "Qwen/Qwen2.5-3B-Instruct"
max_seq_length = 512

print(f"Loading model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,  # Auto detect
    load_in_4bit=True,  # Use 4-bit quantization for T4
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimized
    random_state=42,
)

print("Model loaded with LoRA adapters")
print(f"Trainable parameters: {model.print_trainable_parameters()}")

In [ ]:
# Cell 6: SFT Training with Unsloth
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="sov-grpo-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=max_seq_length,
    dataset_text_field=None,
    packing=False,
)

print("Starting SFT training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 7: Save model
print("Saving model...")
model.save_pretrained("sov-grpo-lora")
tokenizer.save_pretrained("sov-grpo-lora")

# Save as merged model (for Ollama)
print("Saving merged model...")
model.save_pretrained_merged("sov-grpo-merged", tokenizer, save_method="merged_16bit")

print("Models saved!")
print("  LoRA: sov-grpo-lora/")
print("  Merged: sov-grpo-merged/")

In [ ]:
# Cell 8: Test the trained model
print("\n=== TESTING TRAINED MODEL ===")

FastLanguageModel.for_inference(model)

test_questions = [
    "What is the DEFONEOS care floor value?",
    "How many agents are in the BFT council?",
    "A bat and a ball cost $1.10. The bat costs $1.00 more than the ball. How much does the ball cost?",
    "Can you catch a cold from being cold?",
    "Write a Python function is_palindrome(s).",
    "What are the 7 Red Lines?",
    "When does EU AI Act Article 50 take effect?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": "You are SOV33, a sovereign AI model. Answer concisely and accurately."},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=128, use_cache=True, temperature=0.1)
    answer = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nQ: {q}")
    print(f"A: {answer[:200]}")

In [ ]:
# Cell 9: Export for download
import shutil

# Create zip files for download
shutil.make_archive("sov-grpo-lora", "zip", "sov-grpo-lora")
shutil.make_archive("sov-grpo-merged", "zip", "sov-grpo-merged")

print("Zip files created:")
print("  sov-grpo-lora.zip")
print("  sov-grpo-merged.zip")
print("\nDownload these files and extract to use with Ollama.")